# RetailX Portfolio Project: Data Cleaning & ETL Pipeline
### Module 01: Raw Data Ingestion, Audit, Cleaning, and Validation

**Objective**: Ingest dirty raw transactional and catalog data, identify anomalies (duplicates, missing values, extreme outliers, invalid transactions), execute systematic remediation rules, and export standardized datasets to SQLite and CSV.



In [ ]:
import os
import sys
import json
import sqlite3
import pandas as pd
import numpy as np

# Set project paths
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_cleaning import run_pipeline

print("Environment initialized successfully.")



## 1. Raw Data Audit
Let's inspect the raw datasets before cleaning to examine injected real-world imperfections.



In [ ]:
raw_sales = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "sales.csv"))
raw_cust = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "customers.csv"))
raw_prod = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "products.csv"))

print(f"Raw Sales Shape: {raw_sales.shape}")
print(f"Duplicate Transactions: {raw_sales.duplicated(subset=['transaction_id']).sum()}")
print(f"Missing Customer IDs: {raw_sales['customer_id'].isna().sum()}")
print(f"Missing Discounts: {raw_sales['discount'].isna().sum()}")
print(f"Negative Quantities: {(raw_sales['quantity'] <= 0).sum()}")
print(f"Negative Prices: {(raw_sales['unit_price'] <= 0).sum()}")



## 2. Execute Data Cleaning Pipeline
We run the production ETL pipeline from `src/data_cleaning.py`.



In [ ]:
audit_log = run_pipeline()
print(json.dumps(audit_log, indent=2))



## 3. Verify Cleaned Data & Accounting Identities
We confirm that all accounting identities hold true:
- $\text{Revenue} = \text{Quantity} \times \text{Unit Price} \times (1 - \text{Discount})$
- $\text{Profit} = \text{Revenue} - (\text{Quantity} \times \text{Base Cost})$



In [ ]:
clean_sales = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "processed", "sales_clean.csv"))
clean_products = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "processed", "products_clean.csv"))

# Verify zero nulls in critical fields
print("Null count check:")
print(clean_sales[["transaction_id", "date", "quantity", "unit_price", "revenue", "profit"]].isna().sum())

# Verify mathematical accuracy
expected_rev = (clean_sales["quantity"] * clean_sales["unit_price"] * (1 - clean_sales["discount"])).round(2)
diff = (clean_sales["revenue"] - expected_rev).abs().max()
print(f"Max Accounting Discrepancy: ${diff:.4f}")
assert diff < 0.05, "Accounting identity violated!"
print("Accounting verification: PASSED")

